In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
from numpy import dtype
import re
import json
from pathlib import Path

In [3]:
BASE_URL = "https://mktickets.mk"

response = requests.get(BASE_URL)

response.status_code

200

In [4]:
html = response.text
html[522:579]

'<title>MKTickets - Buy Online Tickets : MKTickets</title>'

In [5]:
soup = BeautifulSoup(html, "html.parser")
soup.title

<title>MKTickets - Buy Online Tickets : MKTickets</title>

In [6]:
all_cards = soup.find_all("div")

len(all_cards)

281

In [7]:
all_cards[10].prettify()

'<div class="col-md-6 col-sm-12 col-xs-12 home-top-first home-top-col">\n <div class="slider-wrapper">\n  <div class="event-container">\n   <a class="poster-link" href="https://mktickets.mk/event/lord-of-the-dance/">\n    <img alt="" class="attachment-poster-featured size-poster-featured wp-post-image" decoding="async" fetchpriority="high" sizes="(max-width: 640px) 100vw, 640px" src="https://mktickets.mk/wp-content/uploads/2025/10/mkticks.jpg" srcset="https://mktickets.mk/wp-content/uploads/2025/10/mkticks.jpg 640w, https://mktickets.mk/wp-content/uploads/2025/10/mkticks-250x164.jpg 250w, https://mktickets.mk/wp-content/uploads/2025/10/mkticks-120x79.jpg 120w, https://mktickets.mk/wp-content/uploads/2025/10/mkticks-2x1.jpg 2w"/>\n   </a>\n   <div class="event-hover">\n    <h3>\n     <a href="https://mktickets.mk/event/lord-of-the-dance/">\n      LORD OF THE DANCE\n     </a>\n    </h3>\n    <p>\n     <span>\n      Спортски центар „Јане Сандански”, Скопје,\n     </span>\n     <span>\n   

In [8]:
events_titles = []

for card in all_cards[:]:
    title_tag = card.select_one("div.event-hover > h3")

    if not title_tag:
        continue

    event = {
        "title": title_tag.text.strip()
    }

    events_titles.append(event)

events_titles

[{'title': 'LORD OF THE DANCE'},
 {'title': 'LORD OF THE DANCE'},
 {'title': 'LORD OF THE DANCE'},
 {'title': 'LORD OF THE DANCE'},
 {'title': 'LORD OF THE DANCE'},
 {'title': 'LORD OF THE DANCE'},
 {'title': 'LORD OF THE DANCE'},
 {'title': 'ЈАКОВ ЈОЗИНОВИЌ „Ja za čuda letim“'},
 {'title': 'ЈАКОВ ЈОЗИНОВИЌ „Ja za čuda letim“'},
 {'title': 'THE YOUNG GODS'},
 {'title': 'THE YOUNG GODS'},
 {'title': 'Јован Јованов, Елвир Мекиќ и пријателите'},
 {'title': 'Јован Јованов, Елвир Мекиќ и пријателите'},
 {'title': 'ЈОЦЕ ПАНОВ – МАКЕДОНСКА ФИЛХАРМОНИЈА 2'},
 {'title': 'ЈОЦЕ ПАНОВ – МАКЕДОНСКА ФИЛХАРМОНИЈА 2'},
 {'title': 'LORD OF THE DANCE'},
 {'title': 'LORD OF THE DANCE'},
 {'title': 'ЈАКОВ ЈОЗИНОВИЌ „Ja za čuda letim“'},
 {'title': 'ЈАКОВ ЈОЗИНОВИЌ „Ja za čuda letim“'},
 {'title': 'THE YOUNG GODS'},
 {'title': 'THE YOUNG GODS'},
 {'title': 'Јован Јованов, Елвир Мекиќ и пријателите'},
 {'title': 'Јован Јованов, Елвир Мекиќ и пријателите'},
 {'title': 'ЈОЦЕ ПАНОВ – МАКЕДОНСКА ФИЛХАРМОНИЈА 2'

In [9]:
len(events_titles)

161

In [10]:
df_events_titles = pd.DataFrame(data=events_titles)

df_events_titles

,title
0,LORD OF THE DANCE
1,LORD OF THE DANCE
2,LORD OF THE DANCE
3,LORD OF THE DANCE
4,LORD OF THE DANCE
...,...
156,AUTECHRE
157,AUTECHRE
158,Ezhel Live in Europe 2026
159,Ezhel Live in Europe 2026


In [11]:
unique_events_titles = df_events_titles["title"].unique()

unique_events_titles

array(['LORD OF THE DANCE', 'ЈАКОВ ЈОЗИНОВИЌ „Ja za čuda letim“',
       'THE YOUNG GODS', 'Јован Јованов, Елвир Мекиќ и пријателите',
       'ЈОЦЕ ПАНОВ – МАКЕДОНСКА ФИЛХАРМОНИЈА 2',
       'ТЕАТАР ЗА ДЕЦА И МЛАДИНЦИ – Месечен репертоар',
       'ДРАМСКИ ТЕАТАР – МЕСЕЧЕН РЕПЕРТОАР',
       'Златна Бубамара на Популарноста #29',
       'СИТЕ НАЈУБАВИ НЕШТА 27.02.2026',
       'Македонија – Луксембург  Претквалификации за Евробаскет 2029',
       'FRED WESLEY TRIO', 'Сонуваме Менуваме',
       'Ivan Bosiljčić i Jelena Tomašević 07.03.2026',
       'МЕТАЛКОВЦИ – ПАТУВААТ!', 'Ivan Bosiljčić i Jelena Tomašević',
       'Jelena Tomasevic & Ivan Bosiljčić Битола',
       'Jelena Tomasevic & Ivan Bosiljčić Битола 10.03.2026',
       'ТИХОМИР ПОП АСАНОВИЌ', 'СИТЕ НАЈУБАВИ НЕШТА 13.03.2026',
       'Artist Republic – Регионална конвенција за тетовирање',
       'НАЦИОНАЛЕН ЏЕЗ ОРКЕСТАР СО MIKE HOLOBER', 'Глумците Пеат 24.03',
       'Глумците Пеат 25.03',
       '„Миланова игра“ – концерт во че

In [12]:
len(unique_events_titles)

31

In [13]:
detail_links = []

for card in all_cards:
    a = card.select_one("a.poster-link")
    if a and a.get("href"):

        detail_links.append(a["href"])

detail_links

['https://mktickets.mk/event/lord-of-the-dance/',
 'https://mktickets.mk/event/lord-of-the-dance/',
 'https://mktickets.mk/event/lord-of-the-dance/',
 'https://mktickets.mk/event/lord-of-the-dance/',
 'https://mktickets.mk/event/lord-of-the-dance/',
 'https://mktickets.mk/event/lord-of-the-dance/',
 'https://mktickets.mk/event/jakov-jozinovic-ja-za-cuda-letim/',
 'https://mktickets.mk/event/the-young-gods/',
 'https://mktickets.mk/event/jovan-jovanov-elvir-mekic-i-prijateli/',
 'https://mktickets.mk/event/joce-panov/',
 'https://mktickets.mk/event/lord-of-the-dance/',
 'https://mktickets.mk/event/jakov-jozinovic-ja-za-cuda-letim/',
 'https://mktickets.mk/event/the-young-gods/',
 'https://mktickets.mk/event/jovan-jovanov-elvir-mekic-i-prijateli/',
 'https://mktickets.mk/event/joce-panov/',
 'https://mktickets.mk/event/lord-of-the-dance/',
 'https://mktickets.mk/event/jakov-jozinovic-ja-za-cuda-letim/',
 'https://mktickets.mk/event/the-young-gods/',
 'https://mktickets.mk/event/jovan-jov

In [14]:
df_detail_links = pd.DataFrame(data=detail_links, columns=["href"])

df_detail_links

,href
0,https://mktickets.mk/event/lord-of-the-dance/
1,https://mktickets.mk/event/lord-of-the-dance/
2,https://mktickets.mk/event/lord-of-the-dance/
3,https://mktickets.mk/event/lord-of-the-dance/
4,https://mktickets.mk/event/lord-of-the-dance/
...,...
165,https://mktickets.mk/event/snn13032026/
166,https://mktickets.mk/event/ar/
167,https://mktickets.mk/event/ar/
168,https://mktickets.mk/event/beauty-expo-26/


In [15]:
unique_detail_events = df_detail_links["href"].unique()

unique_detail_events

array(['https://mktickets.mk/event/lord-of-the-dance/',
       'https://mktickets.mk/event/jakov-jozinovic-ja-za-cuda-letim/',
       'https://mktickets.mk/event/the-young-gods/',
       'https://mktickets.mk/event/jovan-jovanov-elvir-mekic-i-prijateli/',
       'https://mktickets.mk/event/joce-panov/',
       'https://mktickets.mk/event/tdm/',
       'https://mktickets.mk/event/dramskiteatar-2/',
       'https://mktickets.mk/event/zbm29/',
       'https://mktickets.mk/event/snn27022026/',
       'https://mktickets.mk/event/makedonija-luxemburg/',
       'https://mktickets.mk/event/fred-wesley-trio/',
       'https://mktickets.mk/event/sonuvame/',
       'https://mktickets.mk/event/ivan-bosiljcic-i-jelena-tomasevic-07-03-2026/',
       'https://mktickets.mk/event/metalkovci/',
       'https://mktickets.mk/event/ibjt/',
       'https://mktickets.mk/event/jelena-tomasevic-ivan-bosiljcic-bitola/',
       'https://mktickets.mk/event/jelena-tomasevic-ivan-bosiljcic-bt-10032026/',
       'ht

In [16]:
len(unique_detail_events)

31

In [17]:
def scrape_event_detail(url):
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to fetch {url}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    def clean_text(text):
        if not text:
         return None

        text = text.replace("\xa0", " ")
        text = text.replace("\n", " ")
        text = text.replace("\r", " ")

        text = re.sub(r"\s+", " ", text)
        return text.strip()

    title_tag = soup.select_one("div#event-details-container.row > :nth-child(2) > h2")
    image_tag = soup.select_one("div.event-main-image > img.wp-post-image")
    if image_tag and image_tag.get("src"):
        image_url = image_tag["src"]
    else:
        image_url = None
    # print(image_url)
    location_tag = soup.select_one("div.event-info > :nth-child(3) > small")
    price_tag = soup.select_one("div.festival-events > ul.tickets > li > p > :nth-child(2)")
    time_start_tag = soup.select_one("div.event-info > :nth-child(5) > :nth-child(2)")
    date_start_tag = soup.select_one("div.event-info > :nth-child(2) > :nth-child(2)")
    city_tag = soup.select_one("div.event-info > :nth-child(4) > :nth-child(2)")
    description_tag = soup.select_one("div.event-content")

    event = {
        "title": clean_text(title_tag.get_text()) if title_tag else None,
        "image_url": image_url if image_url else None,
        "location": clean_text(location_tag.get_text()) if location_tag else None,
        "price": clean_text(price_tag.get_text()).replace("Price:", "").strip() if price_tag else None,
        "time_start": clean_text(time_start_tag.get_text()) if time_start_tag else None,
        "date_start": clean_text(date_start_tag.get_text()) if date_start_tag else None,
        "city": clean_text(city_tag.get_text()) if city_tag else None,
        "description": clean_text(description_tag.get_text()) if description_tag else None,
    }

    return event

In [18]:
events = []

for link in unique_detail_events:
    event = scrape_event_detail(link)

    if event:
        events.append(event)

    len(events)

events

[{'title': 'LORD OF THE DANCE',
  'image_url': 'https://mktickets.mk/wp-content/uploads/2025/10/mkticks.jpg',
  'location': 'Спортски центар „Јане Сандански”',
  'price': '1500-4500ден.',
  'time_start': '20:00',
  'date_start': '07-04-2026',
  'city': 'Скопје',
  'description': '30 години стоечки овации за ирскиот музичко-танцов спектакл! Lord Of The Dance, глобалниот феномен кој го редефинираше сценскиот танц, се враќа во Македонија на 7 април 2026 година со ново ексклузивно шоу за прослава на својата 30-годишнина! Овој значаен настан ветува дека ќе биде грандиозна прослава на трајното наследство на глобалната продукција која плени над 60 милиони обожаватели ширум светот. Не го пропуштајте магичното доживување што во Скопје ќе го приреди светски најпродаваниот танцов спектакл на сите времиња и фантастичната претстава на наградуваниот креатор и режисер Мајкл Флетли која продолжува да руши рекорди и да ја остава публиката без здив.'},
 {'title': 'ЈАКОВ ЈОЗИНОВИЌ „Ja za čuda letim“',
  

In [69]:
# Пример со реден број за да се види колку настани има вкупно.
events_ordering = []

for idx, link in enumerate(unique_detail_events, start=1):
    event = scrape_event_detail(link)
    if event:
        event['number'] = idx
        events.append(event)

events_ordering

[{'title': 'LORD OF THE DANCE',
  'image_url': 'https://mktickets.mk/wp-content/uploads/2025/10/mkticks.jpg',
  'location': 'Спортски центар „Јане Сандански”',
  'price': '1500-4500ден.',
  'time_start': '20:00',
  'date_start': '07-04-2026',
  'city': 'Скопје',
  'description': '30 години стоечки овации за ирскиот музичко-танцов спектакл! Lord Of The Dance, глобалниот феномен кој го редефинираше сценскиот танц, се враќа во Македонија на 7 април 2026 година со ново ексклузивно шоу за прослава на својата 30-годишнина! Овој значаен настан ветува дека ќе биде грандиозна прослава на трајното наследство на глобалната продукција која плени над 60 милиони обожаватели ширум светот. Не го пропуштајте магичното доживување што во Скопје ќе го приреди светски најпродаваниот танцов спектакл на сите времиња и фантастичната претстава на наградуваниот креатор и режисер Мајкл Флетли која продолжува да руши рекорди и да ја остава публиката без здив.',
  'number': 1},
 {'title': 'ЈАКОВ ЈОЗИНОВИЌ „Ja za č

In [20]:
folder = Path("..") / "events-output"
folder.mkdir(parents=True, exist_ok=True)

file_path = folder / "events.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(events, f, ensure_ascii=False, indent=2)

print(f"Saved {len(events)} events to {file_path.resolve()}")

Saved 31 events to D:\Workspace\EventsAggregator-Project\mktickets-scraper\events-output\events.json
